# Notebook 19: the multitask network, warm-started from `surrogate_adme`

The only intervention in this project that has reliably produced a better arm is
**swapping the encoder**. `tab` is TabPFN over `chemprop_medium`; `n15` is the same
recipe over `surrogate_adme` and beats it by +0.011 dev, and `n15` is now our best single
arm on holdout (0.6713) and the best arm we have on CYP2D6.

Every chemprop multitask network we have built warm-starts from `chemprop_medium`,
including `mtl13`, which is a core arm of the live submission. **The encoder swap has
never been applied to the multitask architecture.** That is this notebook, and it is the
one obvious experiment left.

This is notebook 5 with a single substantive change: the warm start is
`surrogate_adme__s42.pt` from `openadmet/chemprop-foundation-pretraining-weights`
instead of `chemprop_medium.pt`. Architecture, 13-head target table, folds, epochs and
checkpointing are byte-identical to the run that produced `mtl13`, so the comparison is
clean.

**Reference to beat**, `mtl13` on the same folds:
`0.5577 / 0.6935 / 0.4515 / 0.8022`, macro dev **0.6262**, macro holdout **0.6652**.

If the swap pays as it did for `tab`→`n15`, expect roughly +0.011 dev. The control is
built in: cell 9 trains warm and cold and prints both, so a broken checkpoint shows up in
two minutes rather than three hours.

**Runtime** about 3 hours. Checkpointed per fold to Drive; Run-all resumes.


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__)
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

NEED = ['meta_train.csv.gz', 'meta_test.csv.gz', 'holdout_mask.npy',
        'scaffold_groups.npy', 'aux_train.npy']
have = sorted(os.listdir('/content/cache'))
missing = [f for f in NEED if f not in have]
assert not missing, (
    'payload is missing %s. Found: %s. Point PAYLOAD_FILE at a payload that has these '
    'five files, or tell Claude and it will ship one inline.' % (missing, have))
print('cache ok:', have)


In [ ]:
!pip -q install chemprop==2.3.1
!pip -q install -U huggingface_hub


In [ ]:
import gc, json, time, urllib.request, numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy'); G=np.load(CACHE/'scaffold_groups.npy',allow_pickle=True)
AUX=np.load(CACHE/'aux_train.npy')
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()

from huggingface_hub import hf_hub_download
# The organisers' fifteen pretrained encoders. nb12 confirmed these load with chemprop's
# own MPNN.load_from_file, so the Multi class below needs no change; it probes the
# fingerprint width dynamically.
CK = hf_hub_download('openadmet/chemprop-foundation-pretraining-weights',
                     'surrogate_adme__s42.pt')
print('checkpoint', round(Path(CK).stat().st_size/1e6,1), 'MB  (surrogate_adme__s42)')

URL=('https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
     '&record_type=datatable&actvty=all&response_type=save&aid=1851')
AID = OUT / 'aid1851.csv'
if not AID.exists():
    print('downloading AID 1851 (first run only; cached on Drive after this)')
    urllib.request.urlretrieve(URL, str(AID))
raw=pd.read_csv(AID, low_memory=False, skiprows=[1])
raw['pac50']=-pd.to_numeric(raw['Fit_LogAC50'], errors='coerce')
PANEL={'p450-cyp1a2':'CYP1A2','p450-cyp2c9':'CYP2C9','p450-cyp2c19':'CYP2C19',
       'p450-cyp2d6':'CYP2D6','p450-cyp3a4':'CYP3A4'}
raw['panel']=raw['Panel Name'].map(PANEL)
raw=raw.dropna(subset=['panel','pac50','PUBCHEM_EXT_DATASOURCE_SMILES'])
print('qHTS measurements', len(raw))


## Build the 13-column target table

Public compounds that are also challenge compounds are dropped, so nothing the blind set
contains leaks in through an auxiliary head.

In [ ]:
def canon(s):
    m=Chem.MolFromSmiles(s) if isinstance(s,str) else None
    return Chem.MolToSmiles(m) if m is not None else None

ours={c for c in map(canon, np.concatenate([SM_TR,SM_TE])) if c}
pub=raw.groupby(['PUBCHEM_EXT_DATASOURCE_SMILES','panel'],as_index=False).pac50.median()
pub['canon']=[canon(s) for s in pub.PUBCHEM_EXT_DATASOURCE_SMILES]
pub=pub.dropna(subset=['canon'])
pub=pub[~pub.canon.isin(ours)]
wide=pub.pivot_table(index='PUBCHEM_EXT_DATASOURCE_SMILES',columns='panel',values='pac50')
PAN=['CYP1A2','CYP2C9','CYP2C19','CYP2D6','CYP3A4']
wide=wide.reindex(columns=PAN)
print('public compounds after dropping overlaps', len(wide))

HEADS=[f'chal_{i}' for i in ISO]+[f'scr_{i}' for i in ISO]+[f'qh_{p}' for p in PAN]
n_ch=len(SM_TR); n_pub=len(wide)
SM_ALL=np.concatenate([SM_TR, wide.index.to_numpy()])
Y=np.full((n_ch+n_pub, len(HEADS)), np.nan, np.float32)
for j,i in enumerate(ISO):
    Y[:n_ch, j]=meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch, 4+j]=AUX[:, j]
Y[n_ch:, 8:]=wide.to_numpy(float)
M=np.isfinite(Y)
# Normalisation from development rows only; using all rows would fit a statistic on
# held-out labels. Affine, so it cannot inflate a Spearman, but it should still be clean.
_norm = Y.copy(); _norm[np.flatnonzero(hold)] = np.nan
mu = np.nanmean(_norm, 0); sd = np.nanstd(_norm, 0)
Yz_nan=((Y-mu)/np.where(sd>0,sd,1.0)).astype(np.float32)  # NaN kept as the mask
print('table', Y.shape, '| labels per head:', dict(zip(HEADS, M.sum(0).tolist())))

In [ ]:
import tempfile
from chemprop.models.model import MPNN

def load_backbone(path, verbose=True):
    """Load one of the organisers' fifteen pretrained encoders.

    They are not complete MPNN checkpoints: their saved hyper_parameters omit keys that
    chemprop's loader assumes are present ('metrics' at minimum), so MPNN.load_from_file
    raises KeyError on them. Three routes are tried in order and the one that worked is
    printed, so a future failure says which stage broke rather than just failing.
    """
    errs = []
    # 1. the plain loader, in case a checkpoint is complete
    try:
        m = MPNN.load_from_file(path)
        if verbose: print('  loaded via MPNN.load_from_file')
        return m
    except Exception as e:
        errs.append(f'load_from_file: {type(e).__name__}: {e}')

    d = torch.load(path, map_location='cpu', weights_only=False)
    hp = d.get('hyper_parameters', None)
    if hp is None:
        raise RuntimeError('checkpoint has no hyper_parameters; tried:\n  ' + '\n  '.join(errs))

    # 2. fill in the optional keys the loader expects, then re-save and load normally.
    #    _load mutates hparams, so it must read from a file rather than this dict.
    try:
        d2 = dict(d); h2 = dict(hp)
        for k, v in [('metrics', None), ('w_t', None), ('batch_norm', False),
                     ('warmup_epochs', 2), ('init_lr', 1e-4), ('max_lr', 1e-3),
                     ('final_lr', 1e-4), ('X_d_transform', None)]:
            h2.setdefault(k, v)
        d2['hyper_parameters'] = h2
        tmp = tempfile.mktemp(suffix='.pt')
        torch.save(d2, tmp)
        m = MPNN.load_from_file(tmp)
        if verbose: print('  loaded after filling missing hyper_parameters:',
                          sorted(set(h2) - set(hp)))
        return m
    except Exception as e:
        errs.append(f'patched hparams: {type(e).__name__}: {e}')

    # 3. the checkpoint is a bare message-passing block: rebuild the model around it
    try:
        from chemprop.nn import BondMessagePassing, MeanAggregation, RegressionFFN
        mp_hp = hp.get('message_passing', hp)
        mp_hp = {k: v for k, v in dict(mp_hp).items() if k != 'cls'}
        mp = BondMessagePassing(**mp_hp)
        sd = d['state_dict']
        inner = {k.split('message_passing.', 1)[1]: v for k, v in sd.items()
                 if 'message_passing.' in k}
        mp.load_state_dict(inner if inner else sd)
        m = MPNN(mp, MeanAggregation(), RegressionFFN(input_dim=mp.output_dim, n_tasks=1))
        if verbose: print('  loaded by rebuilding around BondMessagePassing')
        return m
    except Exception as e:
        errs.append(f'component rebuild: {type(e).__name__}: {e}')

    raise RuntimeError('could not load %s. hyper_parameters keys: %s. state_dict keys: %s\n  %s'
                       % (path, sorted(hp), list(d.get('state_dict', {}))[:8],
                          '\n  '.join(errs)))

print('checkpoint load check:')
_probe = load_backbone(CK)
print('  parameters:', sum(p.numel() for p in _probe.parameters()))
del _probe


In [ ]:
from chemprop import data, featurizers

FZ = featurizers.SimpleMoleculeMolGraphFeaturizer()

# Targets live inside the datapoints, so chemprop's own loader batches them with the
# graphs and nothing has to be re-featurised per minibatch. NaN marks "not measured"
# and becomes the loss mask.
PTS_ALL = [data.MoleculeDatapoint.from_smi(s, Yz_nan[i])
           for i, s in enumerate(SM_ALL)]
PTS_TE = [data.MoleculeDatapoint.from_smi(s, np.zeros(len(HEADS), np.float32))
          for s in SM_TE]
DS_TE = data.MoleculeDataset(PTS_TE, FZ)

def loader(idx=None, bs=128, shuffle=False, test=False):
    pts = PTS_TE if test else ([PTS_ALL[i] for i in idx] if idx is not None else PTS_ALL)
    return data.build_dataloader(data.MoleculeDataset(pts, FZ), num_workers=0,
                                 batch_size=bs, shuffle=shuffle, drop_last=False)

def _fp(m, bmg, V_d, X_d):
    # BatchMolGraph.to() moves in place and returns None, unlike a torch tensor, so it
    # must be called as a statement rather than used as an expression.
    if hasattr(bmg, 'to'):
        bmg.to(DEVICE)
    V_d = V_d.to(DEVICE) if torch.is_tensor(V_d) else V_d
    X_d = X_d.to(DEVICE) if torch.is_tensor(X_d) else X_d
    return m.fingerprint(bmg, V_d, X_d)

class Multi(nn.Module):
    def __init__(self, ckpt, n_out, warm=True):
        super().__init__()
        m = load_backbone(ckpt, verbose=False)
        if not warm:
            for p in m.parameters():
                nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.zeros_(p)
        self.enc = m
        with torch.no_grad():                      # probe on CPU, before .to(DEVICE)
            bmg, V_d, X_d, *_ = next(iter(loader([0, 1, 2, 3], bs=4)))
            d = m.fingerprint(bmg, V_d, X_d).shape[1]
        self.head = nn.Sequential(nn.Linear(d, 512), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(512, n_out))

    def forward(self, bmg, V_d, X_d):
        return self.head(_fp(self.enc, bmg, V_d, X_d))

def train(tr_idx, epochs=25, warm=True, lr=3e-4, bs=128, seed=0):
    torch.manual_seed(seed)
    m = Multi(CK, len(HEADS), warm).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-5)
    ld = loader(tr_idx, bs=bs, shuffle=True)
    for ep in range(epochs):
        m.train()
        for batch in ld:
            bmg, V_d, X_d, Yb, *_ = batch
            t = Yb.to(DEVICE).float()
            msk = torch.isfinite(t)
            if msk.sum() == 0:
                continue
            opt.zero_grad()
            p = m(bmg, V_d, X_d)
            loss = (((p - torch.nan_to_num(t)) ** 2) * msk).sum() / msk.sum()
            loss.backward()
            opt.step()
    return m

@torch.no_grad()
def predict(m, idx=None, bs=256, test=False):
    m.eval()
    out = [m(*batch[:3]).cpu().numpy() for batch in loader(idx, bs=bs, test=test)]
    return np.vstack(out) * sd + mu


## Control first: does the warm start actually load

Two short runs on the same fold, one from the checkpoint and one from a reinitialised
encoder. The warm start should be clearly ahead. If it is not, the checkpoint is not
reaching the weights and everything below is meaningless.

In [ ]:
y1=meta['CYP3A4_pIC50_direct_inhibition'].to_numpy(float)
obs=np.isfinite(y1); dev=np.flatnonzero(obs & ~hold); ho=np.flatnonzero(obs & hold)
pub_idx=np.arange(n_ch, n_ch+n_pub)
quick=np.concatenate([dev[:800], pub_idx[:4000]])
for warm in [True, False]:
    t0=time.time()
    m=train(quick, epochs=6, warm=warm)
    p=predict(m, ho)[:,3]
    print(f'  warm={warm}: CYP3A4 holdout rho {spearmanr(y1[ho],p).statistic:.4f} '
          f'({time.time()-t0:.0f}s for 6 epochs)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

## Full run

Public compounds sit in training for every fold; only the challenge rows rotate.

In [ ]:
EPOCHS = 25
RESULTS = []
y_all = {i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs_any = np.isfinite(np.column_stack([y_all[i] for i in ISO])).any(1)
dev_rows = np.flatnonzero(obs_any & ~hold); ho_rows = np.flatnonzero(obs_any & hold)
pub_idx = np.arange(n_ch, n_ch + n_pub)

# Checkpoint after every fit, so a disconnect costs one fold rather than the whole run.
# Re-running this cell picks up wherever it stopped.
CKPT = OUT / 'nb19_state.npz'
if CKPT.exists():
    st = np.load(CKPT, allow_pickle=True)
    oof_all = st['oof_all']; done = set(st['done'].tolist())
    hp = st['hp'] if st['hp'].size else None
    tp = st['tp'] if st['tp'].size else None
    print('resuming; folds already done:', sorted(done), '| holdout',
          hp is not None, '| test', tp is not None)
else:
    oof_all = np.full((n_ch, 4), np.nan, np.float32); done = set(); hp = tp = None

def save():
    np.savez_compressed(CKPT, oof_all=oof_all, done=np.array(sorted(done)),
                        hp=hp if hp is not None else np.array([]),
                        tp=tp if tp is not None else np.array([]))

t0 = time.time()
for f, (tr, va) in enumerate(GroupKFold(5).split(dev_rows, dev_rows, G[dev_rows])):
    if f in done:
        continue
    m = train(np.concatenate([dev_rows[tr], pub_idx]), epochs=EPOCHS)
    oof_all[dev_rows[va]] = predict(m, dev_rows[va])[:, :4]
    done.add(f); save()
    print(f'  fold {f} done, checkpointed ({time.time()-t0:.0f}s)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

if hp is None:
    m = train(np.concatenate([dev_rows, pub_idx]), epochs=EPOCHS)
    hp = predict(m, ho_rows)[:, :4]; save()
    print(f'  holdout model done ({time.time()-t0:.0f}s)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

if tp is None:
    mt = train(np.concatenate([dev_rows, ho_rows, pub_idx]), epochs=EPOCHS)
    tp = predict(mt, test=True)[:, :4]; save()
    print(f'  test model done ({time.time()-t0:.0f}s)', flush=True)
    del mt; gc.collect(); torch.cuda.empty_cache()

for j, i in enumerate(ISO):
    y = y_all[i]
    d = dev_rows[np.isfinite(y[dev_rows])]; h = ho_rows[np.isfinite(y[ho_rows])]
    np.savez_compressed(OUT / f'mtlsur_{i}.npz', oof=oof_all[d, j], dev=d,
                        hold=hp[np.isin(ho_rows, h), j], ho=h, test=tp[:, j])
    r = float(spearmanr(y[d], oof_all[d, j]).statistic)
    rh = float(spearmanr(y[h], hp[np.isin(ho_rows, h), j]).statistic)
    RESULTS.append(dict(iso=i, dev_spearman=r, holdout_spearman=rh))
    json.dump(RESULTS, open(OUT / 'results_nb19.json', 'w'), indent=2)
    print(f'{i}: dev rho {r:.4f} | holdout rho {rh:.4f}', flush=True)
print('\nmtl13, same architecture on chemprop_medium: 0.5577 0.6935 0.4515 0.8022')
print('  macro dev 0.6262 | macro holdout 0.6652')
print('macro dev %.4f | macro holdout %.4f' % (
    np.mean([x['dev_spearman'] for x in RESULTS]),
    np.mean([x['holdout_spearman'] for x in RESULTS])))

import shutil
shutil.make_archive('/content/nb19_results', 'zip', OUT)
try:
    from google.colab import files
    files.download('/content/nb19_results.zip')
except Exception as e:
    print('download skipped:', type(e).__name__, '- results are on Drive in', OUT)
